In [20]:

import sys
from pathlib import Path
BASE_PATH = Path.cwd().absolute()

sys.path.append(str(BASE_PATH))
sys.path.append(str(BASE_PATH / "Models"))
sys.path.append(str(BASE_PATH / "Functions"))


import HypOpt_CatBoost as Cat
import HypOpt_Gandalf as Gan
import HypOpt_LightGBM as LGBM
import HypOpt_LogisticRegression as Log
import HypOpt_MLP as mlp
import HypOpt_TabNet as Tab
import HypOpt_XGBoost as XGB
import Gandalf as Gan_model
import MLP as MLP_model

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet import TabNetClassifier
import xgboost as xgb

from sklearn.metrics import mean_absolute_error

import pandas as pd
import numpy as np
import torch.nn as nn
import torch
import sqlalchemy as sa
import Results as res
import Get_Data as gd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import  accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import torch

from torch.utils.data import DataLoader
import torch.utils.data as data_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cpu=torch.device("cpu")

ImportError: cannot import name 'TabNetClassifier' from 'pytorch_tabnet' (unknown location)

In [2]:
path="/mnt/c/Users/jrech/OneDrive - Modehaus und Trachtenhaus Rechenauer GmbH/Desktop/Privat/MBA Workflow/Masterarbeit/CODE_BASE/DATEN/DB.db"
engine = sa.create_engine("sqlite:///" + path)
engine.connect()

path_2="/mnt/c/Users/jrech/OneDrive - Modehaus und Trachtenhaus Rechenauer GmbH/Desktop/Privat/MBA Workflow/Masterarbeit/CODE_BASE/DATEN/DB_params.db"
engine_2 = sa.create_engine("sqlite:///" + path_2)
engine_2.connect()

Test= pd.read_sql("SELECT * FROM Test_data_final", engine)
Train= pd.read_sql("SELECT * FROM Train_data_final", engine)


X_test_raw = Test.drop(columns=["verzoegerung_bin_5","w_time","Verzoegerungstage"])
y_test_raw = Test["verzoegerung_bin_5"]
w_test_raw = Test["w_time"]

X_train_raw = Train.drop(columns=["verzoegerung_bin_5","w_time","Verzoegerungstage"])
y_train_raw = Train["verzoegerung_bin_5"]
w_train_raw = Train["w_time"]



In [3]:


def calc_metrics(y_true, y_pred, sample_weight=None, labels=None):
    metrics = {}

    metrics["accuracy_weighted"] = accuracy_score(
        y_true,
        y_pred,
        sample_weight=sample_weight
    )

    metrics["mean_absolute_error_weighted"] = mean_absolute_error(
        y_true,
        y_pred,
        sample_weight=sample_weight
    )

    metrics["f1_weighted"] = f1_score(
        y_true,
        y_pred,
        average="weighted",
        sample_weight=sample_weight,
        zero_division=0
    )


    metrics["precision_weighted"] = precision_score(
        y_true,
        y_pred,
        average="weighted",
        sample_weight=sample_weight,
        zero_division=0
    )

    metrics["recall_weighted"] = recall_score(
        y_true,
        y_pred,
        average="weighted",
        sample_weight=sample_weight,
        zero_division=0
    )

    return metrics



In [4]:


def calc_metrics(y_true, y_pred, sample_weight=None, labels=None):
    metrics = {}

    metrics["accuracy_weighted"] = accuracy_score(
        y_true,
        y_pred,
        sample_weight=sample_weight
    )

    metrics["mean_absolute_error_weighted"] = mean_absolute_error(
        y_true,
        y_pred,
        sample_weight=sample_weight
    )

    metrics["f1_weighted"] = f1_score(
        y_true,
        y_pred,
        average="weighted",
        sample_weight=sample_weight,
        zero_division=0
    )


    metrics["precision_weighted"] = precision_score(
        y_true,
        y_pred,
        average="weighted",
        sample_weight=sample_weight,
        zero_division=0
    )

    metrics["recall_weighted"] = recall_score(
        y_true,
        y_pred,
        average="weighted",
        sample_weight=sample_weight,
        zero_division=0
    )

    return metrics



class ConfidenceIntervalCalculator:
    def __init__(self):
        self.bootstrap_results = pd.DataFrame(columns=[
            "Model",
            "metric",
            "mean_bootstrap",
            "std_bootstrap",
            "ci_lower_95",
            "ci_upper_95"
        ])

        self.bootstrap_samples = pd.DataFrame()
        self.ci_df = None

    def bootstrap_confidence_intervals(
        self,
        model_name,
        y_true,
        y_pred,
        sample_weight=None,
        n_bootstrap=2000,
        alpha=0.05,
        random_state=42):

        rng = np.random.default_rng(random_state)
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
    

    
        sample_weight = np.asarray(sample_weight).astype(float)

        n = len(y_true)
        bootstrap_rows = []

        for _ in range(n_bootstrap):
            idx = rng.choice(n, size=n, replace=True)

            y_true_b = y_true[idx]
            y_pred_b = y_pred[idx]
            w_b = sample_weight[idx]

            metrics_b = calc_metrics(
                y_true_b,
                y_pred_b,
                sample_weight=w_b,
            )

            bootstrap_rows.append(metrics_b)

        bootstrap_df = pd.DataFrame(bootstrap_rows)
        bootstrap_df["Model"] = model_name

        ci_rows = []

        for metric in bootstrap_df.drop(columns=["Model"]).columns:
            values = bootstrap_df[metric].dropna()
            ci_rows.append({
                "Model": model_name,
                "metric": metric,
                "mean_bootstrap": values.mean(),
                "std_bootstrap": values.std(),
                "ci_lower_95": values.quantile(alpha / 2),
                "ci_upper_95": values.quantile(1 - alpha / 2)
            })

        ci_df = pd.DataFrame(ci_rows).round(4)

        self.bootstrap_results = pd.concat(
            [self.bootstrap_results, ci_df],
            ignore_index=True
        )

        self.bootstrap_samples = pd.concat(
            [self.bootstrap_samples, bootstrap_df],
            ignore_index=True
        )

        self.ci_df = ci_df

    

In [5]:
conf=ConfidenceIntervalCalculator()

In [ ]:

Model="CatBoost"
#Cat.Run_Optuna(runs=100, folds=5, 
best_params=pd.read_sql(Model, engine_2).iloc[0].to_dict()
X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw, scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer"))
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)
best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

final_model = CatBoostClassifier(**best_params)

start=time.time()
final_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    sample_weight=w_tr,
    early_stopping_rounds=50,
    verbose=False
)
end=time.time()
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)
traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test

)
res.save_frames(Model, y_pred.flatten(), y_proba ,y_test, w_test, training_time=traing_time)



Results for CatBoost saved successfully at 2026-05-26 09:20.


/tmp/ipykernel_15457/3068902164.py:113: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.bootstrap_results = pd.concat(


In [7]:

Model="Gandalf"
#Gan.Run_Optuna(runs=20,epochs=100, folds=5)

epochs=100
loss_fn = nn.CrossEntropyLoss(reduction="none")
best_params=pd.read_sql(Model, engine_2).iloc[0].to_dict()
X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw, scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer") , feature_output="MLP")
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)


best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

batch_size=best_params.pop("batch_size")

train_dataset = data_utils.TensorDataset(X_tr, y_tr, w_tr)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = data_utils.TensorDataset(X_val, y_val, w_val)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

model = Gan_model.Model(
                    input_dim=X_train.shape[1],
                    gflu_stages=best_params["gflu_stages"],
                    gflu_dropout=best_params["gflu_dropout"],
                    gflu_feature_init_sparsity=best_params["gflu_feature_init_sparsity"],
                    learnable_sparsity=True,
                    )

optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=best_params.pop("weight_decay"))
sheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, min_lr=1e-6)


start=time.time()
accuracay_plot=model.fit(train_dataloader, val_dataloader,epochs, optimizer, sheduler, loss_fn)
end=time.time()

y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)


conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)

traing_time=end-start
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

Epochs 1, Loss: 1.4455, Val Loss: 1.3627, Val Acc: 0.4516
Epochs 2, Loss: 1.3006, Val Loss: 1.3072, Val Acc: 0.4662
Epochs 3, Loss: 1.2371, Val Loss: 1.2589, Val Acc: 0.4927
Epochs 4, Loss: 1.1889, Val Loss: 1.2255, Val Acc: 0.5127
Epochs 5, Loss: 1.1471, Val Loss: 1.1913, Val Acc: 0.5157
Epochs 6, Loss: 1.1109, Val Loss: 1.1676, Val Acc: 0.5289
Epochs 7, Loss: 1.0767, Val Loss: 1.1381, Val Acc: 0.5427
Epochs 8, Loss: 1.0459, Val Loss: 1.1015, Val Acc: 0.5600
Epochs 9, Loss: 1.0157, Val Loss: 1.0951, Val Acc: 0.5609
Epochs 10, Loss: 0.9815, Val Loss: 1.0669, Val Acc: 0.5749
Epochs 11, Loss: 0.9567, Val Loss: 1.0475, Val Acc: 0.5849
Epochs 12, Loss: 0.9290, Val Loss: 1.0246, Val Acc: 0.5971
Epochs 13, Loss: 0.9025, Val Loss: 1.0157, Val Acc: 0.6009
Epochs 14, Loss: 0.8849, Val Loss: 0.9938, Val Acc: 0.6085
Epochs 15, Loss: 0.8624, Val Loss: 0.9874, Val Acc: 0.5962
Epochs 16, Loss: 0.8445, Val Loss: 0.9721, Val Acc: 0.6124
Epochs 17, Loss: 0.8260, Val Loss: 0.9457, Val Acc: 0.6308
Epochs

In [8]:
Model="LightGBM"
#LGBM.Run_Optuna(runs=100, folds=5)

best_params=pd.read_sql(Model, engine_2).iloc[0].to_dict()

X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer"))
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)
best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

model = LGBMClassifier(**best_params)

start=time.time()
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    sample_weight=w_tr,
    eval_sample_weight=[w_val]
)
end=time.time()
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
traing_time=end-start
conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

/home/jrech/micromamba/envs/rapids/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jrech/micromamba/envs/rapids/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Results for LightGBM saved successfully at 2026-05-26 09:26.


In [9]:
Model="Logistic Regression"
#Log.Run_Optuna(runs=100, folds=5)
best_params=pd.read_sql(Model, engine_2).iloc[0].to_dict()

X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer"))
model = LogisticRegression(
    solver=best_params["solver"],
    C=best_params["C"],
    max_iter=best_params["max_iter"],
    class_weight=best_params["class_weight"],
    random_state=42
)

start=time.time()
model.fit(X_train, y_train, sample_weight=w_train)
end=time.time()


X_test = np.nan_to_num(X_test, nan=0)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)

res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

Results for Logistic Regression saved successfully at 2026-05-26 09:26.


In [10]:

Model="Multilayer Perceptron"
#mlp.Run_Optuna(runs=20,epochs=100, folds=3)
epochs=100
best_params=pd.read_sql(Model, engine_2).iloc[0].to_dict()
X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw, scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer") , feature_output="MLP")
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)

best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

batch_size=best_params.pop("batch_size")

train_dataset = data_utils.TensorDataset(X_tr, y_tr, w_tr)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = data_utils.TensorDataset(X_val, y_val, w_val)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


model=MLP_model.Model(input_dim=X_train.shape[1])

optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=best_params.pop("weight_decay"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, min_lr=1e-6)


start=time.time()
accuracay_plot=model.fit(train_dataloader, val_dataloader,epochs, optimizer, scheduler)
end=time.time()

y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)

traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

Fold 1, Loss: 1.5406, Val Loss: 1.2958, Val Acc: 0.4856
Fold 2, Loss: 1.3585, Val Loss: 1.2317, Val Acc: 0.5109
Fold 3, Loss: 1.3075, Val Loss: 1.2031, Val Acc: 0.5158
Fold 4, Loss: 1.2605, Val Loss: 1.1799, Val Acc: 0.5088
Fold 5, Loss: 1.2307, Val Loss: 1.1581, Val Acc: 0.5374
Fold 6, Loss: 1.2025, Val Loss: 1.1378, Val Acc: 0.5452
Fold 7, Loss: 1.1761, Val Loss: 1.1200, Val Acc: 0.5506
Fold 8, Loss: 1.1509, Val Loss: 1.0951, Val Acc: 0.5618
Fold 9, Loss: 1.1378, Val Loss: 1.0727, Val Acc: 0.5664
Fold 10, Loss: 1.1176, Val Loss: 1.0639, Val Acc: 0.5761
Fold 11, Loss: 1.1089, Val Loss: 1.0387, Val Acc: 0.5948
Fold 12, Loss: 1.0817, Val Loss: 1.0301, Val Acc: 0.5892
Fold 13, Loss: 1.0715, Val Loss: 1.0126, Val Acc: 0.5992
Fold 14, Loss: 1.0541, Val Loss: 1.0003, Val Acc: 0.5991
Fold 15, Loss: 1.0374, Val Loss: 0.9875, Val Acc: 0.6163
Fold 16, Loss: 1.0282, Val Loss: 0.9883, Val Acc: 0.6119
Fold 17, Loss: 1.0211, Val Loss: 0.9766, Val Acc: 0.6153
Fold 18, Loss: 1.0052, Val Loss: 0.9591,

In [11]:
Model="TabNet"
#Tab.Run_Optuna(runs=20,epochs=300, folds=3)
best_params=pd.read_sql(Model, engine_2).iloc[0].to_dict()

X_train, X_test, y_train, y_test, w_train, w_test, cat_idxs, cat_dims = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler="none", encoder="ordinal", periodic_transformer=False, feature_output="TabNet")
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)

scaler = best_params.pop("scaler")
encoder = best_params.pop("encoder")
periodic_transformer = best_params.pop("periodic_transformer")
batch_size = best_params.pop("batch_size")
optimizer_params=best_params.pop("lr")
step_size=best_params.pop("step_size")
scheduler_gamma=best_params.pop("scheduler_gamma")

best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")



model = TabNetClassifier(
    **best_params,
    optimizer_params=dict(lr=optimizer_params),
    scheduler_params=dict(step_size=step_size, gamma=scheduler_gamma),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    seed=42,
)

start=time.time()

model.fit(
    X_train=X_tr,
    y_train=y_tr,
    weights=w_tr,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    eval_set=[(X_val, y_val)],
    eval_weights=[w_val],
    eval_metric=['accuracy'],
    max_epochs=300,
    patience=100,
    batch_size=batch_size,
    verbose=0
)
end=time.time()


X_train = np.nan_to_num(X_train, nan=0)
X_test = np.nan_to_num(X_test, nan=0)

y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)

traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)


/home/jrech/micromamba/envs/rapids/lib/python3.12/site-packages/pytorch_tabnet/tab_models/tab_class.py:25: UserWarning: Device used : cuda
  super(TabNetClassifier, self).__post_init__()


epoch 0  | loss: 2.20364 | val_0_accuracy: 0.17655 |  0:00:00s
epoch 1  | loss: 1.46079 | val_0_accuracy: 0.20894 |  0:00:01s
epoch 2  | loss: 1.17957 | val_0_accuracy: 0.23312 |  0:00:01s
epoch 3  | loss: 1.13075 | val_0_accuracy: 0.23175 |  0:00:05s
epoch 4  | loss: 1.08762 | val_0_accuracy: 0.23312 |  0:00:05s
epoch 5  | loss: 1.06833 | val_0_accuracy: 0.2094  |  0:00:06s
epoch 6  | loss: 1.0259  | val_0_accuracy: 0.25456 |  0:00:06s
epoch 7  | loss: 0.94149 | val_0_accuracy: 0.24179 |  0:00:07s
epoch 8  | loss: 0.97524 | val_0_accuracy: 0.224   |  0:00:08s
epoch 9  | loss: 0.96066 | val_0_accuracy: 0.22993 |  0:00:08s
epoch 10 | loss: 0.87507 | val_0_accuracy: 0.24909 |  0:00:09s
epoch 11 | loss: 0.86664 | val_0_accuracy: 0.24544 |  0:00:09s
epoch 12 | loss: 0.8537  | val_0_accuracy: 0.27464 |  0:00:10s
epoch 13 | loss: 0.85143 | val_0_accuracy: 0.27099 |  0:00:10s
epoch 14 | loss: 0.81854 | val_0_accuracy: 0.28878 |  0:00:11s
epoch 15 | loss: 0.81048 | val_0_accuracy: 0.27281 |  0

/home/jrech/micromamba/envs/rapids/lib/python3.12/site-packages/pytorch_tabnet/callbacks/container.py:113: UserWarning: Best weights from best epoch are automatically used!
  callback.on_train_end(logs)


Results for TabNet saved successfully at 2026-05-26 09:30.


In [12]:

Model="XGBoost"
#XGB.Run_Optuna(runs=10,folds=5)
best_params=pd.read_sql(Model, engine_2).iloc[0].to_dict()


scaler = best_params.pop("scaler")
encoder = best_params.pop("encoder")
periodic_transformer = best_params.pop("periodic_transformer")
number_ouf_boost_rounds = best_params.pop("num_boost_round")

X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler=scaler,encoder=encoder,periodic_transformer=periodic_transformer)
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split( X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)



dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
dval   = xgb.DMatrix(X_val, label=y_val, weight=w_val)
dtest  = xgb.DMatrix(X_test, label=y_test, weight=w_test)

start=time.time()
bst = xgb.train(
    best_params,
    dtrain,
    num_boost_round=number_ouf_boost_rounds,
    evals=[(dval, "val")],
    early_stopping_rounds=30,
    verbose_eval=False
)
end=time.time()
traing_time=end-start
y_proba = bst.predict(dtest, output_margin=False)
y_pred = y_proba.argmax(axis=1)
end=time.time()


traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)

res.save_frames(Model, y_pred.astype(str), y_proba ,y_test, w_test, training_time=traing_time)





Results for XGBoost saved successfully at 2026-05-26 09:30.


In [13]:
sqlite_path = "/mnt/c/Users/jrech/OneDrive - Modehaus und Trachtenhaus Rechenauer GmbH/Desktop/Privat/MBA Workflow/Masterarbeit/CODE_BASE/DATEN/DB_results.db"
engine = sa.create_engine("sqlite:///" + sqlite_path)
engine.connect()


df=conf.bootstrap_results.round(4)
df.to_sql("ConfidenceIntervals", con=engine, if_exists="replace", index=False)


35